# 🎮 Epic Games Store - Data Preprocessing (Refined Buckets)

**Objective:** Load, clean, and preprocess the `Epic.csv` data for predicting the time until a game becomes free, using refined time buckets and preparing for an **XGBoost** model.

---

### 📋 **Steps:**
1. Load `Epic.csv`.
2. Clean data (dates, scores, missing values).
3. Calculate `Time_to_Service_Days`.
4. Define **refined categorical `Prediction_Target` bins**.
5. Discuss multi-entry Publisher/Developer strategy.
6. Recommend XGBoost.
7. Update prebaked sentence template.
8. Save preprocessed data.

---

### ⚙️ **Prerequisites**
Ensure `Epic.csv` is in the same directory. Run the cell below to install libraries.

In [ ]:
# Cell 1: Install Libraries
!pip install pandas numpy requests openpyxl xgboost scikit-learn

In [ ]:
# Cell 2: Load and Inspect Data

import pandas as pd
import numpy as np
import os

print(f"Current working directory: {os.getcwd()}")
epic_file = 'Epic.csv'

try:
    df = pd.read_csv(epic_file)
    print(f"\n✅ Successfully loaded '{epic_file}'.")
    
    print("\n--- Initial DataFrame Head ---")
    print(df.head())
    
    print("\n--- Initial DataFrame Info ---")
    df.info()
    
except FileNotFoundError:
    print(f"❌ ERROR: File '{epic_file}' not found.")
    df = None 
except Exception as e:
    print(f"❌ ERROR loading file: {e}")
    df = None

original_columns = list(df.columns) if df is not None else []
print(f"\nOriginal columns: {original_columns}")

### ✨ **Step 3: Data Cleaning and Basic Preprocessing**

Rename columns, handle missing critical values, convert date strings to datetime.

In [ ]:
# Cell 3: Clean Data and Convert Types

if df is not None:
    if not original_columns or len(original_columns) < 7:
        print("Warning: Could not reliably detect original column names. Attempting standard names.")
        if all(col in df.columns for col in ['game_name', 'release_date', 'Added to Service', 'Removed from Service', 'metacritic_score', 'publisher', 'developer']):
             df.rename(columns={'game_name': 'Game', 'release_date': 'Release_Date', 'Added to Service': 'Service_Date', 'Removed from Service': 'Removed_from_Service', 'metacritic_score': 'Metacritic_Score', 'publisher': 'Publisher', 'developer': 'Developer'}, inplace=True)
        else: print("Could not determine correct column names.")
    else: 
        column_mapping = { original_columns[0]: 'Game', original_columns[1]: 'Release_Date', original_columns[2]: 'Service_Date', original_columns[3]: 'Removed_from_Service', original_columns[4]: 'Metacritic_Score', original_columns[5]: 'Publisher', original_columns[6]: 'Developer' }
        valid_mapping = {k: v for k, v in column_mapping.items() if k in df.columns}
        df.rename(columns=valid_mapping, inplace=True)
    
    print("\nRenamed columns attempt complete. Current columns:", list(df.columns))

    if 'Game' in df.columns:
        if df.iloc[0].isnull().all() or pd.isna(df.iloc[0]['Game']):
           df = df.iloc[1:].copy()
           print("Dropped first row assuming it was metadata.")
    else:
        print("Critical 'Game' column not found. Stopping.")
        df = None

if df is not None:
    initial_rows = len(df)
    df.dropna(subset=['Game', 'Service_Date', 'Release_Date'], inplace=True)
    print(f"Dropped {initial_rows - len(df)} rows with missing essential info.")

    df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
    df['Service_Date'] = pd.to_datetime(df['Service_Date'], errors='coerce')

    initial_rows = len(df)
    df.dropna(subset=['Release_Date', 'Service_Date'], inplace=True)
    print(f"Dropped {initial_rows - len(df)} rows with invalid date formats.")

    df['Metacritic_Score'] = pd.to_numeric(df['Metacritic_Score'], errors='coerce').round(0).astype('Int64') 

    print("\n--- Cleaned DataFrame Info ---")
    df.info()
else:
    print("DataFrame not available for cleaning.")

### ⏱️ **Step 4: Calculate Target Variable (`Time_to_Service_Days`)**

In [ ]:
# Cell 4: Calculate Time to Service

if df is not None:
    df['Time_to_Service_Days'] = (df['Service_Date'] - df['Release_Date']).dt.days
    df['Time_to_Service_Days'] = df['Time_to_Service_Days'].apply(lambda x: max(0, x if pd.notna(x) else -1))
    df = df[df['Time_to_Service_Days'] >= 0] 

    print("\n--- Data with Time_to_Service_Days (Head) ---")
    print(df[['Game', 'Release_Date', 'Service_Date', 'Time_to_Service_Days']].head())
else:
    print("DataFrame not available.")

### 🏷️ **Step 5: Define *Refined* Categorical Target Bins (`Prediction_Target`)**

Convert `Time_to_Service_Days` into more granular categories, including 2+ and 4+ year buckets.

In [ ]:
# Cell 5: Create REFINED Prediction Target Bins

if df is not None:
    # Define the refined classification bins 
    # Approximations: 6mo=180d, 18mo=545d, 3yr=1095d, 5yr=1825d
    def classify_time_to_service_refined(days):
        if pd.isna(days):
            return 'Unknown' 
        elif days <= 180:   # Within ~6 Months
            return '0 - 6 Months'
        elif days <= 545:  # Approx 6 to 18 Months
            return '6 - 18 Months' 
        elif days <= 1095: # Approx 1.5 to 3 Years
             return '1.5 - 3 Years'
        elif days <= 1825: # Approx 3 to 5 Years
             return '3 - 5 Years'
        else: # Older than 5 years or never
            return '5+ Years / Never'

    df['Prediction_Target'] = df['Time_to_Service_Days'].apply(classify_time_to_service_refined)

    print("\n--- Data with REFINED Prediction Target Bins (Head) ---")
    print(df[['Game', 'Time_to_Service_Days', 'Prediction_Target']].head())
    print("\nDistribution of REFINED Target Bins:")
    print(df['Prediction_Target'].value_counts())
else:
    print("DataFrame not available.")

### 👥 **Step 6: Handling Multi-Entry Publishers/Developers**

**Strategy Reminder (Future Step):** Combine all datasets, explode Publisher/Developer, calculate aggregate metrics (Avg Time, Total Count), merge numerical features back.

**For now, kept as strings.**

### 🤖 **Step 7: Model Recommendation (XGBoost)**

**XGBoost (Extreme Gradient Boosting)** remains the recommended model due to its performance on tabular data, ability to handle missing values, and regularization features. You'll use `xgboost.XGBClassifier` after feature engineering.

### 📝 **Step 8: Updated Prebaked Sentence Template**

Reflecting the model's use of Metacritic and publisher history:

```text
Based on the publisher ([Publisher Name(s)]) which historically averages around [Publisher_Avg_TTS / 365:.1f] years to bring its games ([Publisher_Sum_Inclusions] total) to services, and considering the game's Metacritic score of [Metacritic Score] (which typically influences wait times), the model predicts the likelihood of '[Game Name]' appearing on [Target Service Name] as:

**Prediction:** [Prediction_Target (e.g., 1.5 - 3 Years)] 
(Confidence: [Model Confidence Score:.1f]%)
```

### 💾 **Step 9: Save Preprocessed Data**

In [ ]:
# Cell 6: Save Processed Data

if df is not None:
    output_filename = 'Epic_Preprocessed_Refined.csv'
    columns_to_save = [
        'Game', 'Release_Date', 'Service_Date', 'Metacritic_Score', 
        'Publisher', 'Developer', 'Time_to_Service_Days', 'Prediction_Target' 
    ]
    columns_to_save = [col for col in columns_to_save if col in df.columns]

    df[columns_to_save].to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n✅ Preprocessed Epic Games data saved to: {output_filename}")
    print(f"Total rows saved: {len(df)}")
else:
    print("DataFrame not available. Nothing to save.")

---
**Next Steps:**
1. Repeat preprocessing for `Xbox.csv` and `PS.csv` using this refined logic.
2. Combine the three `_Preprocessed_Refined.csv` files.
3. Perform detailed Feature Engineering.
4. Train an `XGBoostClassifier` model.
---